# 23 AG markets and technologies

**Purpose.** To see what policy-driven SMR deployment does to the rest of the power system, this
notebook reads protocol §8 (signed 2026-10-28) on the same pairs as the ledger: the six stage-1 pairs
and the 308 primary stage-2 pairs of notebook 24 (314 pairs). It reports four things, each as run − baseline (Δ):

1. **Market share.** Capacity, generation, new builds and retirements by technology group, and
   transmission capacity, national, 2026–2050.
2. **Value factors of new builds.** The value of a new MW per MWh relative to the system-average
   value per MWh, from `valnew`, with the formulas of `postprocessing/bokehpivot/reeds2.py:538-578`.
3. **Existing-fleet value.** Market revenue per MWh and per kW of each technology group.
4. **Stage-2 spread.** The median and range of the main quantities across the stage-2 runs.

5. **Hourly energy value** (§6, amendment A17). Capture prices by technology group and the price profile by
   season and hour-ending, for the six stage-1 pairs.

**What it writes** (`results/exports/23/`): `mk_levels.parquet`, `mk_delta.parquet`, `mk_vf.parquet`,
`mk_vf_delta.parquet`, `mk_fleet_value.parquet`, `mk_tran.csv`, `mk_stage1_2050.csv`,
`mk_stage2_summary.csv`, `mk_hourly_capture.parquet`, `mk_hourly_capture_delta.parquet`,
`mk_hourly_capture_2035_2050.csv`, `mk_hourly_profile.parquet`, `mk_hourly_profile_delta.parquet`,
`mk_hourly_profile_2050.csv`. It opens no ReEDS outputs file and states no hypothesis. Observations go to
`docs/observations.md` by hand (protocol §10).


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("23")
E = L.EXTRACTS_AG
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
GROUPS = [g for g in L.GROUPS]
Y0, Y1 = 2026, 2050
USD = L.TO2025_FROM_2004

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


## 1. The pairs

Stage 1 is the six P50 mandate runs against `fbR_none`. Stage 2 is every primary pair of notebook 24
(`exports/24/ag2_pairs.csv`): under the cap, the primary cap baseline `bx_aj_p75_f50_co2cap` only.

In [2]:
s1 = pd.DataFrame(dict(stage="1", batch="smr100", case=[f"smr100_{t}_p50" for t in TARGETS], target=TARGETS,
                       condition="reference", base_batch="itcfb", baseline="fbR_none", primary=True, mandate=True,
                       difference="programme"))
P2 = pd.read_csv(paths.exports("24") / "ag2_pairs.csv")
PAIRS = pd.concat([s1, P2[P2["primary"]]], ignore_index=True)
print(PAIRS.groupby("stage").agg(pairs=("case", "size")))
assert len(PAIRS) == 6 + 60 + 178 + 30 + 40
RUNS = pd.concat([PAIRS[["batch", "case"]],
                  PAIRS[["base_batch", "baseline"]].set_axis(["batch", "case"], axis=1)]).drop_duplicates()
print("distinct runs read:", len(RUNS))

       pairs
stage       
1          6
2a        60
2b       178
2c        30
2d        40
distinct runs read: 322


## 2. Market share by technology group

The groups are those of the ledger (`aglib.ledger.group`, protocol §4.5). Units: capacity in GW,
generation in TWh, new builds and retirements in GW per year (`cap_new_ann_nat` and `ret_ann_nat` are
the MW of a solve period divided by its length in years). The generation share is the group's
generation over the sum of all groups. Storage generation is discharge. A group's cumulative new
builds and retirements over 2026–2050 apply each period's annual rate to the calendar years of that
period that fall in 2026–2050.

In [3]:
def read(key, batches=None):
    fr = []
    for b in sorted(set(RUNS["batch"]) if batches is None else batches):
        f = E / b / f"{key}.parquet"
        if f.exists():
            d = pd.read_parquet(f)
            fr.append(d[d["case"].isin(RUNS.loc[RUNS["batch"] == b, "case"])].assign(batch=b))
    d = pd.concat(fr, ignore_index=True)
    d["t"] = d["t"].astype(int)
    return d


def by_group(key, col):
    d = read(key)
    d = d.assign(group=d["i"].map(L.group))
    return d.groupby(["batch", "case", "group", "t"])["Value"].sum().rename(col)


LV = pd.concat([by_group("cap_nat", "cap_MW"), by_group("gen_ann_nat", "gen_MWh"),
                by_group("cap_new_ann_nat", "new_MW_per_yr"), by_group("ret_ann_nat", "ret_MW_per_yr")],
               axis=1).fillna(0.0).reset_index()
SOLVE = read("pvf_onm")[["batch", "case", "t"]].drop_duplicates()     # every solve year of every run
LV = LV[LV["t"] >= 2023]
tot = LV.groupby(["batch", "case", "t"])[["cap_MW", "gen_MWh"]].transform("sum")
LV["gen_share"] = LV["gen_MWh"] / tot["gen_MWh"]
LV["cap_share"] = LV["cap_MW"] / tot["cap_MW"]


def cumulative(d, col):
    """Sum of an annual rate over the calendar years 2026-2050, period by period."""
    # the period of solve year t starts after the run's previous solve year (not the group's previous row)
    sy = SOLVE.groupby(["batch", "case"])["t"].apply(lambda s: dict(zip(sorted(s)[1:], sorted(s)[:-1])))
    prev = pd.Series([sy[(b, c)].get(t, t - 1) for b, c, t in zip(d["batch"], d["case"], d["t"])], index=d.index)
    yrs = (np.minimum(d["t"], Y1) - np.maximum(prev, Y0 - 1)).clip(lower=0)
    return (d[col] * yrs).groupby([d["batch"], d["case"], d["group"]]).sum()


CUM = pd.concat([cumulative(LV, "new_MW_per_yr").rename("new_MW_2026_2050"),
                 cumulative(LV, "ret_MW_per_yr").rename("ret_MW_2026_2050")], axis=1).reset_index()
LV = LV[LV["t"] >= Y0]
LV.to_parquet(OUT / "mk_levels.parquet", index=False)
print(LV.shape, "level rows;", LV["case"].nunique(), "runs")
chk = LV.groupby(["case", "t"])["gen_share"].sum()
assert np.allclose(chk, 1.0), "generation shares sum to one"
display(LV.query("case == 'fbR_none' and t in [2026, 2035, 2050]")
          .pivot_table(index="group", columns="t", values=["cap_MW", "gen_share"]).round(3))

(37329, 10) level rows; 322 runs


cap_MW                              gen_share              
t                               2026           2035          2050      2026   2035   2050
group                                                                                    
Canadian imports         8149.714844    9187.761719  8.800427e+03     0.011  0.011  0.008
H2 and DAC production            NaN     310.471008  6.408500e+01       NaN  0.000  0.000
coal                   153088.109375   57952.316406  5.246109e+04     0.132  0.072  0.005
existing nuclear        97636.703125   96800.898438  9.410700e+04     0.171  0.139  0.108
gas                    568619.187500  671969.687500  8.465656e+05     0.377  0.307  0.340
hydro                   79482.406250   83210.546875  8.332064e+04     0.063  0.054  0.043
other                    7821.700195   12188.936523  1.357810e+04     0.007  0.013  0.012
storage                 97658.265625  135691.187500  1.267136e+05    -0.005 -0.006 -0.005
wind + solar           425961.656250  841652.250000  1.232215e+06     0.243  0.412  0.490

The differences. Each row is one pair, one group and one solve year; `run` and `base` are levels and
`delta` = run − base. Cumulative new builds and retirements are added as `t = 0` rows.

In [4]:
MEAS = ["cap_MW", "gen_MWh", "gen_share", "cap_share", "new_MW_per_yr", "ret_MW_per_yr"]
lv = LV.set_index(["batch", "case", "group", "t"])[MEAS]
cum = CUM.assign(t=0).set_index(["batch", "case", "group", "t"])
keys = ["stage", "batch", "case", "target", "condition", "baseline"]


def deltas(levels, meas):
    rows = []
    long = levels.reset_index().melt(id_vars=["batch", "case", "group", "t"], value_vars=meas, var_name="measure",
                                     value_name="v")
    for _, p in PAIRS.iterrows():
        a = long[(long["batch"] == p["batch"]) & (long["case"] == p["case"])]
        b = long[(long["batch"] == p["base_batch"]) & (long["case"] == p["baseline"])]
        m = a.drop(columns=["batch", "case"]).merge(b.drop(columns=["batch", "case"]), on=["group", "t", "measure"],
                                                    how="outer", suffixes=("_run", "_base")).fillna(0.0)
        rows.append(m.assign(**{k: p[k] for k in keys}))
    d = pd.concat(rows, ignore_index=True).rename(columns={"v_run": "run", "v_base": "base"})
    d["delta"] = d["run"] - d["base"]
    return d


DL = pd.concat([deltas(lv, MEAS), deltas(cum, ["new_MW_2026_2050", "ret_MW_2026_2050"])], ignore_index=True)
DL.to_parquet(OUT / "mk_delta.parquet", index=False)
print(DL.shape)

(225124, 12)


**Stage 1 in 2050.** Δ generation share (percentage points), Δ capacity (GW) and Δ cumulative
retirements over 2026–2050 (GW), by group, for the six P50 pairs.

In [5]:
s1d = DL[DL["stage"] == "1"]
def s1_table(meas, t, scale):
    x = s1d[(s1d["measure"] == meas) & (s1d["t"] == t)].pivot_table(index="group", columns="target", values="delta")
    return (x.reindex(index=GROUPS, columns=TARGETS) * scale).round(2)
new_smr = s1d[(s1d["measure"] == "new_MW_2026_2050") & (s1d["group"] == "SMR (new)")].set_index("target")["delta"]
cap_smr = s1d[(s1d["measure"] == "cap_MW") & (s1d["t"] == Y1) & (s1d["group"] == "SMR (new)")].set_index("target")["delta"]
print("SMR: cumulative new builds 2026-2050 against capacity in 2050 (GW):")
display(pd.DataFrame({"new builds": new_smr, "capacity 2050": cap_smr}).reindex(TARGETS).mul(1e-3).round(2))
assert np.allclose(new_smr.reindex(TARGETS), cap_smr.reindex(TARGETS), rtol=1e-3), "SMR builds add up to the 2050 fleet"
T1 = {"d gen share (pts), 2050": s1_table("gen_share", Y1, 100), "d capacity (GW), 2050": s1_table("cap_MW", Y1, 1e-3),
      "d retirements 2026-2050 (GW)": s1_table("ret_MW_2026_2050", 0, 1e-3),
      "d new builds 2026-2050 (GW)": s1_table("new_MW_2026_2050", 0, 1e-3)}
for k, v in T1.items():
    print(k); display(v)
pd.concat(T1, names=["quantity"]).to_csv(OUT / "mk_stage1_2050.csv")

SMR: cumulative new builds 2026-2050 against capacity in 2050 (GW):


,new builds,capacity 2050
target,,
eia,20.7,20.7
aj,36.4,36.4
iaea,75.7,75.7
mck,103.4,103.4
cop28,203.4,203.4
eo,303.4,303.4


d gen share (pts), 2050


target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
SMR (new),2.36,4.15,8.65,11.82,23.20,34.46
existing nuclear,0.01,0.02,0.03,0.03,0.03,0.03
gas,-1.04,-1.38,-3.03,-4.70,-11.71,-19.03
coal,-0.00,-0.07,-0.07,-0.07,-0.08,-0.10
wind + solar,-1.32,-2.73,-5.57,-7.08,-11.47,-15.38
hydro,0.00,0.01,0.01,0.01,0.03,0.00
storage,0.01,0.02,0.03,0.04,0.07,0.09
Canadian imports,0.00,0.00,0.00,0.00,0.00,0.00
H2 and DAC production,0.00,0.00,0.00,0.00,0.00,0.00


d capacity (GW), 2050


target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
SMR (new),20.70,36.40,75.70,103.40,203.40,303.40
existing nuclear,0.00,0.00,0.00,0.00,0.00,0.00
gas,-15.77,-25.41,-55.25,-76.28,-154.78,-222.64
coal,-0.21,-0.77,-0.90,-1.26,-2.80,-4.79
wind + solar,-35.29,-70.51,-141.67,-176.31,-284.59,-375.72
hydro,-0.00,-0.00,-0.01,-0.01,-0.03,-0.03
storage,-2.01,-3.32,-5.28,-8.45,-17.50,-26.77
Canadian imports,0.00,0.00,0.00,0.00,0.00,0.00
H2 and DAC production,-0.00,0.13,0.11,0.05,0.06,0.00


d retirements 2026-2050 (GW)


target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
SMR (new),0.00,0.00,0.00,0.00,0.00,0.00
existing nuclear,0.00,0.00,0.00,0.00,0.00,0.00
gas,0.30,0.05,2.05,1.85,1.83,0.62
coal,0.32,0.60,0.73,1.16,2.84,4.32
wind + solar,0.00,0.00,0.00,0.00,0.00,0.00
hydro,0.00,0.00,0.00,0.00,0.00,0.00
storage,0.00,-0.21,-2.08,-2.30,-3.17,-3.64
Canadian imports,0.00,0.00,0.00,0.00,0.00,0.00
H2 and DAC production,0.00,0.00,0.02,0.02,0.03,0.08


d new builds 2026-2050 (GW)


target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
SMR (new),20.70,36.40,75.70,103.40,203.40,303.40
existing nuclear,0.00,0.00,0.00,0.00,0.00,0.00
gas,-18.18,-27.83,-58.19,-79.52,-159.38,-228.30
coal,0.10,-0.18,-0.58,-0.66,-0.94,-2.03
wind + solar,-35.29,-70.51,-141.67,-176.31,-284.59,-375.72
hydro,-0.00,-0.00,-0.01,-0.01,-0.03,-0.03
storage,-2.01,-3.53,-7.36,-10.75,-20.67,-30.42
Canadian imports,0.00,0.00,0.00,0.00,0.00,0.00
H2 and DAC production,0.00,0.13,0.13,0.07,0.08,0.08


**Transmission.** Interzonal capacity summed over lines (`tran_out`, MW, by line type) and MW-miles
(`tran_mi_out`). Δ by pair and year.

In [6]:
tr = pd.concat([read("tran_out").assign(measure="MW"), read("tran_mi_out").assign(measure="MW-mi")])
tr = tr[tr["t"] >= Y0].groupby(["batch", "case", "measure", "t"])["Value"].sum()
rows = []
for _, p in PAIRS.iterrows():
    a = tr.xs((p["batch"], p["case"]), level=["batch", "case"])
    b = tr.xs((p["base_batch"], p["baseline"]), level=["batch", "case"])
    m = pd.concat([a.rename("run"), b.rename("base")], axis=1).fillna(0.0).reset_index()
    rows.append(m.assign(delta=m["run"] - m["base"], **{k: p[k] for k in keys}))
TR = pd.concat(rows, ignore_index=True)
TR.to_csv(OUT / "mk_tran.csv", index=False)
display(TR[(TR["stage"] == "1") & (TR["t"] == Y1)].pivot_table(index="measure", columns="target", values="delta")
        .reindex(columns=TARGETS).map(lambda v: f"{v:,.0f}"))

target,eia,aj,iaea,mck,cop28,eo
measure,,,,,,
MW,"-14,002","-24,705","-47,597","-53,925","-69,802","-84,037"
MW-mi,"-3,322,304","-5,046,432","-10,812,192","-12,226,784","-16,821,616","-20,553,392"


## 3. Value factors of new builds

`valnew` reports, for each technology and zone with new MW in a solve year, the value of that year's new
capacity in the energy market (`val_load`), the capacity market (`val_resmarg`) and the RPS market
(`val_rps`), and its MWh. The benchmark rows give the same values for the load itself, by zone and for
the system (`sys`). Following `reeds2.py:538-578`, at the national level:

- value factor = (Σ value / Σ MWh of the technology) ÷ (system benchmark value / benchmark MWh);
- the energy and capacity parts use `val_load` and `val_resmarg` alone in both terms.

Notebook 23 sums each technology's zones, then the technologies of a group. A value factor exists only
for a technology-year with new MW > 0 (protocol §8); a row with MWh ≤ 0 (storage that charges more
than it discharges) gets no value factor and is counted. Values are 2004 $ in the extracts; the ratio
has no dollar year, and the $/MWh columns are 2025 $.

In [7]:
vn = read("valnew")
vn = vn.rename(columns={"*": "measure", "*.1": "i", "*.2": "r"})
bench = vn[(vn["i"] == "benchmark") & (vn["r"] == "sys")].pivot_table(index=["batch", "case", "t"], columns="measure",
                                                                       values="Value")
tech = vn[vn["i"] != "benchmark"].assign(group=lambda d: d["i"].map(L.group))
tech = tech.pivot_table(index=["batch", "case", "group", "t"], columns="measure", values="Value", aggfunc="sum").fillna(0.0)
tech = tech[tech["MW"] > 0]
VAL = ["val_load", "val_resmarg", "val_rps"]
b = bench.reindex(tech.index.droplevel("group"))
b.index = tech.index
VF = pd.DataFrame(index=tech.index)
VF["new_MW"] = tech["MW"]
VF["MWh"] = tech["MWh"]
ok = tech["MWh"] > 0
VF["vf"] = (tech[VAL].sum(axis=1) / tech["MWh"]) / (b[VAL].sum(axis=1) / b["MWh"])
VF["vf_load"] = (tech["val_load"] / tech["MWh"]) / (b["val_load"] / b["MWh"])
VF["vf_resmarg"] = (tech["val_resmarg"] / tech["MWh"]) / (b["val_resmarg"] / b["MWh"])
VF["value_per_MWh_2025"] = tech[VAL].sum(axis=1) / tech["MWh"] * USD
VF["bench_value_per_MWh_2025"] = b[VAL].sum(axis=1) / b["MWh"] * USD
VF.loc[~ok, ["vf", "vf_load", "vf_resmarg", "value_per_MWh_2025"]] = np.nan
VF = VF.reset_index()
VF = VF[VF["t"] >= Y0]
VF.to_parquet(OUT / "mk_vf.parquet", index=False)
print(f"{len(VF)} group-years with new MW; {int((~ok).sum())} with MWh <= 0 (no value factor)")
display(VF.query("case == 'fbR_none' and t in [2035, 2050]").pivot_table(index="group", columns="t", values="vf").round(3))

24509 group-years with new MW; 775 with MWh <= 0 (no value factor)


t,2035,2050
group,,
coal,0.803,NaN
gas,1.292,1.172
hydro,0.888,NaN
other,0.977,1.123
storage,0.858,1.438
wind + solar,0.731,0.552


Δ value factor for each pair and group-year in which both runs build the group. A group-year that
only one of the two runs builds is listed with a missing Δ, so the count of such years is visible.

In [8]:
vf = VF.set_index(["batch", "case", "group", "t"])
rows = []
for _, p in PAIRS.iterrows():
    a = vf.xs((p["batch"], p["case"]), level=["batch", "case"])[["vf", "vf_load", "vf_resmarg", "new_MW"]]
    bb = vf.xs((p["base_batch"], p["baseline"]), level=["batch", "case"])[["vf", "vf_load", "vf_resmarg", "new_MW"]]
    m = a.join(bb, how="outer", lsuffix="_run", rsuffix="_base").reset_index()
    for c in ["vf", "vf_load", "vf_resmarg"]:
        m[f"d_{c}"] = m[f"{c}_run"] - m[f"{c}_base"]
    rows.append(m.assign(**{k: p[k] for k in keys}))
VFD = pd.concat(rows, ignore_index=True)
VFD.to_parquet(OUT / "mk_vf_delta.parquet", index=False)
both = VFD["d_vf"].notna()
print(f"{both.sum()} group-years built in both runs; {(~both).sum()} in one run only")
display(VFD[(VFD["stage"] == "1") & (VFD["t"] == Y1)].pivot_table(index="group", columns="target", values="d_vf")
        .reindex(columns=TARGETS).round(3))

20301 group-years built in both runs; 3877 in one run only


target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
gas,0.027,0.063,0.128,0.202,0.793,2.390
other,-0.051,-0.073,-0.089,-0.085,-0.063,-0.087
storage,0.012,0.053,0.061,0.104,0.220,0.261
wind + solar,-0.002,-0.007,-0.000,0.017,0.079,0.082


## 4. Existing-fleet value

Market revenue per MWh and per kW-year of each technology group: Σ `revenue_nat` (energy, capacity,
RPS and storage charging, by category) ÷ Σ generation (`gen_ann_nat`), and ÷ Σ capacity (`cap_nat`).
The groups hold every vintage, so "gas" includes new gas plants and "existing nuclear" includes the four
exogenous units (A9). 2025 $.

Check: for the one-technology group "existing nuclear" (`nuclear`), the revenue per MWh equals the sum
over categories of ReEDS's own `revenue_en_nat`. For wind and solar ReEDS divides by uncurtailed
energy, so the check uses nuclear only.

In [9]:
rev = read("revenue_nat").assign(group=lambda d: d["i"].map(L.group))
rev = rev[rev["t"] >= Y0].pivot_table(index=["batch", "case", "group", "t"], columns="rev_cat", values="Value",
                                      aggfunc="sum").fillna(0.0)
lvg = LV.set_index(["batch", "case", "group", "t"])
FV = rev.join(lvg[["gen_MWh", "cap_MW"]], how="left")
cats = [c for c in ["load", "res_marg", "rps", "charge"] if c in FV]
FV["rev_total"] = FV[cats].sum(axis=1)
for c in cats + ["rev_total"]:
    FV[f"{c}_per_MWh_2025"] = FV[c] / FV["gen_MWh"].where(FV["gen_MWh"] > 0) * USD
    FV[f"{c}_per_kWyr_2025"] = FV[c] / (FV["cap_MW"].where(FV["cap_MW"] > 0) * 1e3) * USD
FV = FV.reset_index()
FV.to_parquet(OUT / "mk_fleet_value.parquet", index=False)

ren = read("revenue_en_nat")
ren = ren[(ren["i"] == "nuclear") & (ren["t"] >= Y0)].groupby(["batch", "case", "t"])["Value"].sum() * USD
mine = FV[FV["group"] == "existing nuclear"].set_index(["batch", "case", "t"])["rev_total_per_MWh_2025"]
m = pd.concat([mine.rename("ledger"), ren.rename("reeds")], axis=1).dropna()
rel = ((m["ledger"] - m["reeds"]).abs() / m["reeds"].abs().clip(lower=1e-9)).max()
print(f"existing nuclear $/MWh against revenue_en_nat: {len(m)} run-years, largest relative difference {rel:.1e}")
assert rel < 1e-4

existing nuclear $/MWh against revenue_en_nat: 3864 run-years, largest relative difference 2.5e-07


In [10]:
rows = []
fv = FV.set_index(["batch", "case", "group", "t"])
VCOL = ["rev_total_per_MWh_2025", "load_per_MWh_2025", "res_marg_per_MWh_2025", "rev_total_per_kWyr_2025"]
for _, p in PAIRS.iterrows():
    a = fv.xs((p["batch"], p["case"]), level=["batch", "case"])[VCOL]
    bb = fv.xs((p["base_batch"], p["baseline"]), level=["batch", "case"])[VCOL]
    d = (a - bb).reset_index().melt(id_vars=["group", "t"], var_name="measure", value_name="delta")
    rows.append(d.assign(**{k: p[k] for k in keys}))
FVD = pd.concat(rows, ignore_index=True)
FVD.to_parquet(OUT / "mk_fleet_value_delta.parquet", index=False)
print("Stage 1, 2050: d revenue per MWh (2025 $/MWh)")
display(FVD[(FVD["stage"] == "1") & (FVD["t"] == Y1) & (FVD["measure"] == "rev_total_per_MWh_2025")]
        .pivot_table(index="group", columns="target", values="delta").reindex(index=GROUPS, columns=TARGETS).round(2))

Stage 1, 2050: d revenue per MWh (2025 $/MWh)


target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
SMR (new),NaN,NaN,NaN,NaN,NaN,NaN
existing nuclear,-0.37,-1.76,-2.84,-4.32,-9.84,-12.79
gas,0.20,-1.38,-1.27,-2.12,-0.64,12.95
coal,-0.55,23.74,22.50,17.18,9.95,11.66
wind + solar,-0.31,-1.30,-1.67,-2.31,-4.63,-6.64
hydro,-0.48,-1.71,-2.11,-2.96,-8.04,-10.92
storage,NaN,NaN,NaN,NaN,NaN,NaN
Canadian imports,-1.57,-4.02,-5.66,-6.59,-10.46,-10.48
H2 and DAC production,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Stage-2 spread

The median and range across the runs of each stage and target, for a fixed list of quantities: Δ
generation share in 2050, Δ cumulative retirements and new builds over 2026–2050, Δ value factor in 2050
(group-years built in both runs) and Δ revenue per MWh in 2050. Stage 2b is grouped by condition, and its
Δ contains the condition and the programme together (no baseline exists in each condition).

In [11]:
spec = [("gen_share", Y1, DL, "measure", "delta", 100, "d gen share 2050 (pts)"),
        ("ret_MW_2026_2050", 0, DL, "measure", "delta", 1e-3, "d retirements 2026-2050 (GW)"),
        ("new_MW_2026_2050", 0, DL, "measure", "delta", 1e-3, "d new builds 2026-2050 (GW)"),
        ("rev_total_per_MWh_2025", Y1, FVD, "measure", "delta", 1.0, "d revenue per MWh 2050 (2025 $)")]
parts = []
for meas, t, d, mcol, vcol, sc, lab in spec:
    x = d[(d[mcol] == meas) & (d["t"] == t)]
    parts.append(x.assign(quantity=lab, value=x[vcol] * sc)[["stage", "target", "condition", "group", "quantity", "value"]])
x = VFD[VFD["t"] == Y1].dropna(subset=["d_vf"])
parts.append(x.assign(quantity="d value factor 2050", value=x["d_vf"])[["stage", "target", "condition", "group", "quantity", "value"]])
SP = pd.concat(parts, ignore_index=True)
SUM = SP.groupby(["quantity", "stage", "condition", "target", "group"])["value"].agg(["median", "min", "max", "size"]).reset_index()
SUM.to_csv(OUT / "mk_stage2_summary.csv", index=False)
show = SUM[(SUM["stage"] == "2a") & (SUM["quantity"] == "d gen share 2050 (pts)")]
display(show.pivot_table(index="group", columns="target", values="median").reindex(index=GROUPS, columns=TARGETS).round(2))

target,eia,aj,iaea,mck,cop28,eo
group,,,,,,
SMR (new),2.36,4.15,8.66,11.82,23.18,34.44
existing nuclear,0.00,0.02,0.03,0.03,0.03,0.03
gas,-1.12,-1.36,-3.08,-4.80,-11.75,-18.88
coal,-0.00,-0.07,-0.07,-0.07,-0.08,-0.11
wind + solar,-1.23,-2.72,-5.54,-6.98,-11.41,-15.48
hydro,-0.00,0.01,0.01,0.01,0.03,0.01
storage,0.01,0.03,0.03,0.04,0.07,0.09
Canadian imports,0.00,0.00,0.00,0.00,0.00,0.00
H2 and DAC production,0.00,0.00,0.00,0.00,0.00,0.00


## 6. Hourly energy value (stage 1, amendment A17)

Notebook 20 §7 sums, per solve year, season and hour-ending of the representative timeslices, the load
payment and quantity and each group's output and its energy value at its zone's load price
(`derived_hourly.parquet`; stage-1 runs and baselines only). From these:

- **Capture price** of a group = energy value ÷ output (2025 $/MWh): what one MWh of that group's output
  earned in the energy market. Capacity and RPS revenue are not in it (they are in §4).
- **Load-weighted price** = load payment ÷ load quantity (2025 $/MWh).
- **Energy value factor** = capture price ÷ load-weighted price, of the same run and year.
- **Profile**: the load-weighted price and each group's share of output in each (season, hour-ending) cell.

Check: the capture price of "existing nuclear" equals the load revenue per MWh of §4 (`revenue_nat`
`load` ÷ `gen_ann_nat`) in every stage-1 run-year.

In [12]:
HR = pd.concat([pd.read_parquet(E / b / "derived_hourly.parquet").assign(batch=b)
                for b in ["smr100", "itcfb"]], ignore_index=True)
HR = HR[HR["case"].isin(list(s1["case"]) + ["fbR_none"])]
ann = HR.groupby(["batch", "case", "t", "item"])["Value"].sum().unstack("item")
rows = []
for g in GROUPS:
    if f"gen_MWh|{g}" not in ann:
        continue
    mwh, val = ann[f"gen_MWh|{g}"], ann[f"gen_val|{g}"]
    rows.append(pd.DataFrame({"group": g, "gen_MWh": mwh, "capture_2025": val / mwh.where(mwh > 0) * USD}))
CAP = pd.concat(rows).reset_index()
lw = (ann["load_pay"] / ann["load_MWh"] * USD).rename("load_wtd_2025").reset_index()
CAP = CAP.merge(lw, on=["batch", "case", "t"])
CAP["value_factor"] = CAP["capture_2025"] / CAP["load_wtd_2025"]
CAP.to_parquet(OUT / "mk_hourly_capture.parquet", index=False)

chk = CAP[CAP["group"] == "existing nuclear"].set_index(["batch", "case", "t"])["capture_2025"]
ref = FV[FV["group"] == "existing nuclear"].set_index(["batch", "case", "t"])["load_per_MWh_2025"]
m = pd.concat([chk.rename("hourly"), ref.rename("fleet")], axis=1, join="inner").dropna()
rel = ((m["hourly"] - m["fleet"]).abs() / m["fleet"].abs().clip(lower=1e-9)).max()
print(f"existing nuclear capture price against section 4: {len(m)} run-years, largest relative difference {rel:.1e}")
assert len(m) > 0 and rel < 1e-4

existing nuclear capture price against section 4: 84 run-years, largest relative difference 1.2e-07


In [13]:
cb = CAP.set_index(["case", "group", "t"])
base = cb.xs("fbR_none", level="case")
D = []
for _, p in s1.iterrows():
    a = cb.xs(p["case"], level="case")
    d = (a[["capture_2025", "load_wtd_2025", "value_factor"]] - base[["capture_2025", "load_wtd_2025", "value_factor"]]
         .reindex(a.index)).reset_index()
    D.append(d.assign(target=p["target"], case=p["case"]))
CAPD = pd.concat(D, ignore_index=True).rename(columns={"capture_2025": "d_capture_2025",
                                                       "load_wtd_2025": "d_load_wtd_2025", "value_factor": "d_value_factor"})
CAPD.to_parquet(OUT / "mk_hourly_capture_delta.parquet", index=False)
t50 = CAP[CAP["t"] == Y1].pivot_table(index="group", columns="case", values="value_factor").reindex(GROUPS).dropna(how="all")
print("energy value factor, 2050 (capture price / load-weighted price)")
display(t50[["fbR_none"] + list(s1["case"])].round(3))
out = CAP[CAP["t"].isin([2035, Y1])].merge(CAPD[["case", "group", "t", "d_capture_2025", "d_value_factor"]],
                                         on=["case", "group", "t"], how="left")
out.to_csv(OUT / "mk_hourly_capture_2035_2050.csv", index=False)

energy value factor, 2050 (capture price / load-weighted price)


case,fbR_none,smr100_eia_p50,smr100_aj_p50,smr100_iaea_p50,smr100_mck_p50,smr100_cop28_p50,smr100_eo_p50
group,,,,,,,
SMR (new),NaN,1.076,1.058,0.988,0.994,0.976,0.975
existing nuclear,0.984,0.981,0.977,0.970,0.965,0.949,0.947
gas,1.063,1.064,1.063,1.066,1.065,1.086,1.151
coal,0.966,0.964,0.988,0.987,0.985,0.976,0.980
wind + solar,0.803,0.806,0.815,0.814,0.813,0.813,0.782
hydro,1.020,1.025,1.033,1.035,1.037,1.013,1.025
Canadian imports,0.908,0.904,0.897,0.871,0.863,0.843,0.833
other,1.222,1.208,1.194,1.224,1.230,1.263,1.269


In [14]:
cell = HR.groupby(["case", "t", "season", "hod", "item"])["Value"].sum().unstack("item").fillna(0.0)
gm = cell[[c for c in cell if c.startswith("gen_MWh|")]]
PR = pd.DataFrame({"load_wtd_2025": cell["load_pay"] / cell["load_MWh"] * USD, "load_MWh": cell["load_MWh"]})
for g in ["SMR (new)", "existing nuclear", "gas", "wind + solar", "storage"]:
    if f"gen_MWh|{g}" in gm:
        PR[f"share|{g}"] = gm[f"gen_MWh|{g}"] / gm.sum(axis=1)
PR = PR.reset_index()
PR.to_parquet(OUT / "mk_hourly_profile.parquet", index=False)
pb = PR[PR["case"] == "fbR_none"].set_index(["t", "season", "hod"])
PD = []
for c in s1["case"]:
    a = PR[PR["case"] == c].set_index(["t", "season", "hod"])
    PD.append((a.drop(columns="case") - pb.drop(columns="case").reindex(a.index)).reset_index().assign(case=c))
PRD = pd.concat(PD, ignore_index=True)
PRD.to_parquet(OUT / "mk_hourly_profile_delta.parquet", index=False)
pd.concat([PR[PR["t"] == Y1].assign(kind="level"), PRD[PRD["t"] == Y1].assign(kind="delta")]).to_csv(
    OUT / "mk_hourly_profile_2050.csv", index=False)
print("2050, EO P50 minus fbR_none: load-weighted price by season and hour-ending (2025 $/MWh)")
display(PRD[(PRD["t"] == Y1) & (PRD["case"] == "smr100_eo_p50")].pivot_table(
    index="hod", columns="season", values="load_wtd_2025").round(2))

2050, EO P50 minus fbR_none: load-weighted price by season and hour-ending (2025 $/MWh)


season,fall,spring,summer,winter
hod,,,,
3,-12.53,-14.020000,-14.12,-10.77
6,-12.95,-16.309999,-15.66,-11.58
9,-11.14,-14.960000,-12.20,-11.02
12,-9.35,-6.020000,-3.79,-12.98
15,-7.29,-2.650000,-3.08,-9.12
18,-10.36,-7.860000,-7.66,-10.72
21,-11.10,-11.050000,-8.88,-10.81
24,-11.68,-13.350000,-10.62,-10.28


## 7. What reads these exports

The H3 review page (notebook 25) and the observation log. Nothing in the ledger reads them.